# 09 — F1 reorder alerts

Compute reorder_point and suggested_qty per SKU × DC using organic run rate + lead time + elasticity-driven safety stock. Output alert list.

**Upstream:**
- `clean_demand_weekly.parquet` — per-week base-unit qty per (SKU × channel × DC). Used to pool across channels and compute a DC-level run rate + std.
- `inv_snapshot.parquet` — today's `Available` and `On Hand` per (SKU × DC). `Available = On Hand − Allocated`.
- `item_master.parquet` — Lead Time (freeform string, needs parsing), Case Pack, Country of Origin, Manufacturer.

**Core math** (per SKU × DC):
- `run_rate_wk = mean(weekly qty summed across channels)`
- `std_wk     = std(weekly qty summed across channels)`
- `lead_time_wk = parse(item_master.Lead Time) || 13 wk default` (upper bound on ranges, bare number = months)
- `safety_stock = 1.65 × std_wk × sqrt(lead_time_wk)` (classic 95% service level under roughly-normal demand)
- `reorder_point = run_rate_wk × lead_time_wk + safety_stock`
- `reorder_flag = available_now < reorder_point`
- `suggested_qty = max(0, reorder_point + 4·run_rate_wk − available_now)` rounded up to case pack.

**Confidence**: `high` if ≥ 8 clean weeks AND lead-time was parsed AND inv `Available` not null. `medium` if 2 of 3. `low` otherwise.

**Output:**
- `reorder_alerts.parquet` — full table, one row per (SKU × DC).
- `reorder_alerts.csv` — same data for hand-off to the UI.

**Deliberate omissions (v1):**
- No elasticity-based safety stock (used CV-derived std instead — see notes/status.md).
- MOQ is not enforced (too messy: "2,000 LB", "Ocean container size"). Shown as a note column.
- Lead-time variance from PO history is a separate next-feature, not v1.

**Promotes to:** `src/reorder.py`.

## 1. Imports

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'pipeline' else Path.cwd()
ART = ROOT / 'pipeline' / 'artifacts'
ART.mkdir(parents=True, exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.reorder import (
    SERVICE_LEVEL_Z,
    FORWARD_COVER_WEEKS,
    DEFAULT_LEAD_WEEKS,
    MIN_WEEKS_FOR_HIGH,
    MIN_POS_FOR_LEAD,
    build_reorder_alerts,
    compute_dc_stats,
    compute_lead_time_from_po,
    parse_case_pack,
    parse_lead_time_weeks,
    prepare_item_master,
)

DC_MAP = {'1': 'SF', '2': 'NJ', '3': 'LA'}

## 2. Load upstream

In [2]:
weekly = pd.read_parquet(ART / 'clean_demand_weekly.parquet')
inv    = pd.read_parquet(ART / 'inv_snapshot.parquet')
im     = pd.read_parquet(ART / 'item_master.parquet')
po     = pd.read_parquet(ART / 'po.parquet')

# Build TOTAL weekly outflow (raw sales incl. promos + markdowns) for reorder
# math. Why not clean_demand_weekly: the warehouse drains at total outflow,
# not cleaned demand. T-32206 SF: clean ≈ 5k/wk but raw ≈ 9k/wk — the gap
# systematically eats safety stock every lead cycle. Validated in backtest
# (notebook 10): switching to total_weekly lifts p90 recall 83% → 100% and
# keeps simulated on-hand above zero. Clean demand stays the input for
# forecasts / elasticity where we want to exclude known-cause demand spikes.
sales = pd.read_parquet(ART / 'sales.parquet')
sales['QTY_BASE']   = sales['QUANTITY_adj'].astype(float) * sales['QTYBSUOM'].fillna(1).astype(float)
sales['DC']         = sales['LOCNCODE'].astype(str).map(DC_MAP)
sales               = sales.dropna(subset=['DC'])
sales['week_start'] = pd.to_datetime(sales['DOCDATE']).dt.to_period('W-SUN').dt.start_time
total_weekly = (sales.groupby(['ITEMNMBR', 'DC', 'week_start'], as_index=False)['QTY_BASE']
                     .sum()
                     .rename(columns={'QTY_BASE': 'qty_base'}))

print(f'clean_demand_weekly : {weekly.shape}  '
      f'({weekly["ITEMNMBR"].nunique()} SKU × {weekly["DC"].nunique()} DC × '
      f'{weekly["week_start"].nunique()} weeks)')
print(f'total_weekly (raw)  : {total_weekly.shape}  '
      f'({total_weekly["ITEMNMBR"].nunique()} SKU × {total_weekly["DC"].nunique()} DC × '
      f'{total_weekly["week_start"].nunique()} weeks)')
print(f'inv_snapshot        : {inv.shape}')
print(f'item_master         : {im.shape}')
print(f'po                  : {po.shape}')

clean_demand_weekly : (35591, 8)  (83 SKU × 3 DC × 172 weeks)
total_weekly (raw)  : (22997, 4)  (83 SKU × 3 DC × 172 weeks)
inv_snapshot        : (219, 5)
item_master         : (65, 15)
po                  : (5281, 16)


## 3. Do the work

In [3]:
# Sanity-peek the upstream lookups that feed the alert builder before we
# collapse it all into one call. Useful for buyer-facing "where did that
# come from" questions and for diagnosing low-confidence rows later.
im_p = prepare_item_master(im)
print('Lead-time parse sample (item master):')
print(im_p[['ITEMNMBR', 'Lead Time', 'lead_time_wk', 'lead_parsed']].head(10).to_string())
print(f'\nparsed ok : {int(im_p["lead_parsed"].sum())} / {len(im_p)}')
print(f'null      : {int(im_p["Lead Time"].isna().sum())}')

po_dc, po_sku = compute_lead_time_from_po(po, DC_MAP)
print(f'\nPO-history lead coverage (≥{MIN_POS_FOR_LEAD} POs per lane): {len(po_dc)} lanes')
print(po_dc[po_dc['ITEMNMBR'] == 'T-32206'][['ITEMNMBR', 'DC', 'lead_time_days_med', 'n_pos', 'lead_time_wk_po']].to_string())
print(f'\nPO-history pooled lead (≥2 POs per SKU across DCs): {len(po_sku)} SKUs')

# Run the alert builder on TOTAL weekly outflow — this is the production feed.
dc_stats = compute_dc_stats(total_weekly)
print(f'\ndc_stats (total): {dc_stats.shape}')

alerts = build_reorder_alerts(total_weekly, inv, im, po=po, dc_map=DC_MAP)
print(f'\nreorder_alerts: {alerts.shape}')
print(f'rows flagged for reorder : {int(alerts["reorder_flag"].sum())} / {len(alerts)}')
print(f'confidence counts        : {alerts["confidence"].value_counts().to_dict()}')
print(f'lead_time_source         : {alerts["lead_time_source"].value_counts().to_dict()}')
alerts.head(10)

Lead-time parse sample (item master):
   ITEMNMBR Lead Time  lead_time_wk  lead_parsed
0   A-61220       NaN           NaN        False
1   A-61243       NaN           NaN        False
2  A-61260N       NaN           NaN        False
3   A-61280       NaN           NaN        False
4   A-61012  3 months         12.99         True
5  AC-B16BK       NaN           NaN        False
6   AC-B4BK       NaN           NaN        False
7   AC-B8BK       NaN           NaN        False
8  AC-B3SLJ       NaN           NaN        False
9  AC-B6SLJ       NaN           NaN        False

parsed ok : 51 / 65
null      : 14

PO-history lead coverage (≥3 POs per lane): 156 lanes
    ITEMNMBR  DC  lead_time_days_med  n_pos  lead_time_wk_po
147  T-32206  LA                19.0     23         2.714286
148  T-32206  NJ                36.0     51         5.142857
149  T-32206  SF                20.0     30         2.857143

PO-history pooled lead (≥2 POs per SKU across DCs): 59 SKUs

dc_stats (total): (233, 9)

,ITEMNMBR,Description,DC,vendor,country,run_rate_wk,std_wk,cv,n_clean_weeks,lead_time_wk,...,reorder_point,safety_stock,reorder_flag,suggested_qty,suggested_cases,case_pack,MOQ,first_week,last_week,confidence
0,A-61117,AM GSG Candy 1 lb / Bag,NJ,GL,China,1202.731034,1250.006363,1.039307,145,16.285714,...,27910.711132,8323.377141,True,32736.000000,1488.0,22.0,NaN,2023-01-02,2025-12-29,high
1,A-61280,Am GSG Rt Tea 80 Bags (US Costco),SF,ST,USA,1619.714286,1545.283959,0.954047,21,13.000000,...,30249.426621,9193.140907,True,36728.283764,NaN,1.0,NaN,2023-01-02,2025-09-29,medium
2,AC-B9SL,POP AM GSG Root Slices 9 oz,SF,POP,USA,176.062500,315.879612,1.794133,16,13.000000,...,4168.030727,1879.218227,True,4920.000000,82.0,60.0,NaN,2023-01-02,2025-11-17,medium
3,AC-B9SLE,NaN,SF,NaN,NaN,2843.323529,1445.151239,0.508261,34,13.000000,...,45560.641257,8597.435375,True,56933.935375,NaN,NaN,NaN,2023-01-09,2025-11-24,medium
4,F-04220,POP Ginger Chews Original 2 oz (24ct/cs),NJ,SNA,Indonesia,1593.400000,1435.287682,0.900770,30,12.990000,...,29233.736732,8535.470732,True,35607.336732,NaN,1.0,40HQ,2024-09-23,2025-12-29,high
5,F-04220,POP Ginger Chews Original 2 oz (24ct/cs),SF,SNA,Indonesia,743.111111,390.023059,0.524852,18,12.990000,...,11972.430165,2319.416832,True,14944.874610,NaN,1.0,40HQ,2024-09-23,2025-12-22,high
6,F-04221,POP Ginger Chews Lemon 2 oz (24ct/cs),NJ,SNA,Indonesia,1703.000000,1552.621125,0.911698,29,12.990000,...,31355.207581,9233.237581,True,38167.207581,NaN,1.0,40HQ,2024-09-23,2025-12-29,high
7,F-04221,POP Ginger Chews Lemon 2 oz (24ct/cs),SF,SNA,Indonesia,867.187500,525.791621,0.606318,16,12.990000,...,14391.580584,3126.814959,True,17860.330584,NaN,1.0,40HQ,2024-09-23,2025-12-22,high
8,F-12418,Ferrero 24 Pcs Floorstand (48),LA,Ferrero USA,Poland,15.027778,18.841423,1.253773,36,2.857143,...,95.485406,52.548898,True,192.000000,4.0,48.0,"2,000LB",2023-01-30,2026-01-12,high
9,F-12998,Ferrero Rocher 48's Flat Gift Box,LA,Ferrero USA,Poland,130.212766,247.288342,1.899110,94,2.642857,...,1007.455784,663.322046,True,1536.000000,96.0,16.0,"2,000LB",2023-01-02,2026-03-16,high


## 4. Validate

In [4]:
# ---- Shape / null checks -----------------------------------------------------
assert len(alerts) == alerts[['ITEMNMBR', 'DC']].drop_duplicates().shape[0], \
    'duplicate (ITEMNMBR, DC) rows'
assert alerts['reorder_point'].notna().all(), 'null reorder_point'
assert (alerts['safety_stock'] >= 0).all(), 'negative safety_stock'

print('=== Top 15 reorder-flagged rows by weeks_of_cover ===')
cols = ['ITEMNMBR', 'Description', 'DC', 'available_now', 'run_rate_wk',
        'weeks_of_cover', 'reorder_point', 'suggested_qty', 'confidence']
print(alerts[alerts['reorder_flag']].head(15)[cols].to_string(index=False))

# ---- Reorder-flag distribution by DC ---------------------------------------
print('\n=== reorder_flag by DC ===')
print(alerts.groupby('DC')['reorder_flag'].agg(['sum', 'count']).to_string())

print('\n=== confidence by flag ===')
print(alerts.groupby(['reorder_flag', 'confidence']).size().unstack(fill_value=0).to_string())

# ---- T-32206 spot check ----------------------------------------------------
print('\n=== T-32206 (Tiger Balm Patch Warm) — showcase SKU ===')
spot = alerts[alerts['ITEMNMBR'] == 'T-32206'][cols + [
    'lead_time_wk', 'lead_time_source', 'std_wk', 'safety_stock',
    'case_pack', 'suggested_cases',
]]
print(spot.to_string(index=False))

# SF is our known stockout lane (the pipeline earlier showed on_hand dipping to
# -17k base units in 2023). Sanity-check the numbers:
#   - Lead Time in item_master is "Half a year or more" -> 26 weeks (parsed)
#   - run_rate should land near 4k-15k/week per DC (we saw that in step 07)
#   - safety_stock should be a meaningful fraction of run_rate * lead_time
t_sf = alerts[(alerts['ITEMNMBR'] == 'T-32206') & (alerts['DC'] == 'SF')].iloc[0]
print(f'\n--- T-32206 SF manual trace ---')
print(f'  run_rate_wk     : {t_sf["run_rate_wk"]:>10,.1f} u/wk')
print(f'  std_wk          : {t_sf["std_wk"]:>10,.1f} u/wk')
print(f'  lead_time_wk    : {t_sf["lead_time_wk"]:>10,.1f}  ({t_sf["lead_time_source"]})')
print(f'  demand@lead     : {t_sf["run_rate_wk"]*t_sf["lead_time_wk"]:>10,.0f} u')
print(f'  safety_stock    : {t_sf["safety_stock"]:>10,.0f} u  (= 1.65 * std * sqrt(lead))')
print(f'  reorder_point   : {t_sf["reorder_point"]:>10,.0f} u')
print(f'  available_now   : {t_sf["available_now"]:>10,.0f} u')
print(f'  weeks_of_cover  : {t_sf["weeks_of_cover"]:>10,.2f} wk')
print(f'  reorder_flag    : {bool(t_sf["reorder_flag"])}')
print(f'  suggested_qty   : {t_sf["suggested_qty"]:>10,.0f} u '
      f'({t_sf["suggested_cases"]:.1f} cases of {int(t_sf["case_pack"])})')

=== Top 15 reorder-flagged rows by weeks_of_cover ===
ITEMNMBR                                  Description DC  available_now  run_rate_wk  weeks_of_cover  reorder_point  suggested_qty confidence
 A-61117                      AM GSG Candy 1 lb / Bag NJ            0.0  1202.731034        0.000000   2.791071e+04   3.273600e+04       high
 A-61280            Am GSG Rt Tea 80 Bags (US Costco) SF            0.0  1619.714286        0.000000   3.024943e+04   3.672828e+04     medium
 AC-B9SL                  POP AM GSG Root Slices 9 oz SF            0.0   176.062500        0.000000   4.168031e+03   4.920000e+03     medium
AC-B9SLE                                          NaN SF            0.0  2843.323529        0.000000   4.556064e+04   5.693394e+04     medium
 F-04220     POP Ginger Chews Original 2 oz (24ct/cs) NJ            0.0  1593.400000        0.000000   2.923374e+04   3.560734e+04       high
 F-04220     POP Ginger Chews Original 2 oz (24ct/cs) SF            0.0   743.111111        0.

## 5. Save downstream artifact

In [5]:
alerts.to_parquet(ART / 'reorder_alerts.parquet')
print(f'wrote parquet: {ART / "reorder_alerts.parquet"}  ({alerts.shape})')

# CSV hand-off for the UI. Explicit encoding + quoting rules so the
# buyer-facing consumer can parse without surprises.
csv_path = ART / 'reorder_alerts.csv'
alerts.to_csv(csv_path, index=False)
print(f'wrote csv    : {csv_path}')

wrote parquet: /Users/johnpork/repos/pop_prompt2/pipeline/artifacts/reorder_alerts.parquet  ((233, 27))
wrote csv    : /Users/johnpork/repos/pop_prompt2/pipeline/artifacts/reorder_alerts.csv


## 6. Promote

Once validation above looks right, extract the core logic into `src/reorder.py` and replace the inline code here with `from src.<module> import ...`. Downstream dev notebooks can then import the same function.